# ML-08: Capstone Modeling Lane

**Lane:** Content-opportunity scoring — predicting `actual_ctr_march` from position, impressions,
intent, search volume, and query count. Built on `FlyRank/internship-warehouse`, March 2026.

**This notebook must beat, on the same rows and the same metric, the Week 4 baseline**: predict
the position-band's average CTR (bucket-average). Metric: **MAE**, continuing Week 3/4's choice
for direct comparability, not Precision@K (which would better suit a triage-ranking use case, but
was set aside here to keep this comparison apples-to-apples with the frozen baseline).


## 1) Method Choice and Why

**Method: Random Forest regressor.**

The Week 4 baseline already models one variable (position) via bucket-averaging. The real
question for a "first upgrade" is what bucket-averaging can't do: combine **position, impressions,
intent, search volume, and query count together**, and capture the non-linear CTR-vs-position
curve already seen in the Week 4 bucket table (0.34% / 0.34% / 0.29% / 0.24% / 0.13% — not linear).

Logistic Regression was ruled out (built for classification, not continuous CTR). K-Means was
ruled out (unsupervised, doesn't predict a target). Between a single Decision Tree and a Random
Forest: this dataset has real noise a single tree would overfit to — a documented 37% tie cluster
of zero-click items (Week 4) and ties between the 1-3 and 4-6 position bands. A Random Forest's
averaging across many trees, each seeing a different bootstrap sample, smooths out that noise
rather than memorizing it. Interpretability is reduced versus a single tree, but stability matters
more given how noisy the target is — this is the smallest method that meaningfully answers the
question, not the largest one available.


## 2) Split Design

**Grouped split by `client_hash_id`, not a random row split.**

This week's session demonstrated the exact failure mode this dataset is exposed to: a model
scored perfectly on a random split, then collapsed once the split was changed to hold out entire
clients — "nothing about the model changed... only the question we were asking it." Every row in
this feature frame carries a `client_hash_id`, and many content items belong to the same client.
A random row split lets the model see some of a client's items in training and others in test,
so it can learn that client's typical CTR level rather than the general position-to-CTR
relationship — a leak the deployment question (predicting for content the model has never seen
performance for) would never allow in practice.

To make this concrete rather than just asserted, the notebook below trains the same Random Forest
under **both** a naive random split and a grouped-by-client split, and reports both MAEs side by
side. The gap between them is the demonstration, not just the claim.


In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    );
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"


In [ ]:
feature_query = f"""
    WITH march_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_march,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march
        FROM read_parquet('{MARCH_PATH}')
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        m.client_hash_id, m.content_hash_id,
        m.avg_position_march, m.impressions_march,
        d.main_intent, d.search_volume, d.category_count,
        m.clicks_march * 1.0 / NULLIF(m.impressions_march, 0) AS actual_ctr_march
    FROM march_agg m
    JOIN read_parquet('{CONTENT_PATH}') d ON m.content_hash_id = d.content_hash_id
"""
df = con.sql(feature_query).df().dropna(subset=["actual_ctr_march", "avg_position_march", "main_intent"])
print(df.shape)
df.head()


**Correction:** the original feature list included `content_visible_query_count`, carried over incorrectly from Week 3. That column lives in `fact_content_query_90d` (the sealed test table), not `dim_content`, and using it here would have meant depending on a table this notebook isn't supposed to touch for development. It's replaced with `category_count` (from `dim_content` — how many category tags a content item has), caught from the query's own error message rather than assumed.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

numeric_features = ["avg_position_march", "impressions_march", "search_volume", "category_count"]
categorical_features = ["main_intent"]

def build_pipeline():
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("pre", pre),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])

X = df[numeric_features + categorical_features]
y = df["actual_ctr_march"]
groups = df["client_hash_id"]

# --- Naive random row split (the trap the session warns about) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
model_random = build_pipeline()
model_random.fit(X_train_r, y_train_r)
mae_random_split = mean_absolute_error(y_test_r, model_random.predict(X_test_r))

# --- Grouped split by client_hash_id (the correct deployment question) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
model_grouped = build_pipeline()
model_grouped.fit(X_train_g, y_train_g)
mae_grouped_split = mean_absolute_error(y_test_g, model_grouped.predict(X_test_g))

print(f"MAE, naive random split:   {mae_random_split:.5f}")
print(f"MAE, grouped-by-client split: {mae_grouped_split:.5f}")
print(f"\nGap: {mae_grouped_split - mae_random_split:.5f} "
      f"({(mae_grouped_split / mae_random_split - 1) * 100:+.1f}% worse under the honest split)")


**Expected pattern, matching the session's own demonstration:** the random split should look
better than the grouped split, because it lets the model partially learn each client's baseline
CTR level rather than the general position-to-CTR relationship. The grouped-split number is the
one used for every comparison from here on — the random-split number exists only to show why it
would have been the wrong one to report.


## 3) Train and Compare Against the Week 4 Baseline

Same rows (the grouped-split test set), same metric (MAE). The Week 4 baseline predicts the
position-band's average CTR, frozen from Week 4's own bucket table — it does not get to see this
test set's data before making its prediction, the same way the Random Forest doesn't.


In [ ]:
# Week 4 baseline, frozen — the exact bucket-average table from Week 4's Signal 1 check
expected_ctr_by_band = {
    "1-3": 0.0034, "4-6": 0.0034, "7-10": 0.0029, "11-20": 0.0024, "21+": 0.0013
}

def position_band(pos):
    if pos <= 3: return "1-3"
    elif pos <= 6: return "4-6"
    elif pos <= 10: return "7-10"
    elif pos <= 20: return "11-20"
    else: return "21+"

X_test_g = X_test_g.copy()
X_test_g["position_band"] = X_test_g["avg_position_march"].apply(position_band)
baseline_preds = X_test_g["position_band"].map(expected_ctr_by_band)

mae_baseline = mean_absolute_error(y_test_g, baseline_preds)
mae_model = mae_grouped_split  # already computed on the identical grouped test set

print(f"Week 4 baseline MAE (bucket-average, same test rows): {mae_baseline:.5f}")
print(f"Random Forest MAE (grouped split, same test rows):    {mae_model:.5f}")

if mae_model < mae_baseline:
    improvement = (1 - mae_model / mae_baseline) * 100
    print(f"\nRandom Forest beats the baseline by {improvement:.1f}% lower MAE.")
else:
    shortfall = (mae_model / mae_baseline - 1) * 100
    print(f"\nRandom Forest does NOT beat the baseline — {shortfall:.1f}% worse MAE.")
    print("Report this honestly rather than reframing the comparison to hide it.")


## 4) Errors and Interpretation

Two views: which features the model actually leans on (permutation importance, not the
model's built-in importances, since built-in importances can be biased toward high-cardinality
numeric features), and what the worst individual errors look like.


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model_grouped, X_test_g[numeric_features + categorical_features],
                               y_test_g, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df


In [ ]:
# Worst individual errors on the grouped test set
X_test_g["actual_ctr"] = y_test_g.values
X_test_g["predicted_ctr"] = model_grouped.predict(X_test_g[numeric_features + categorical_features])
X_test_g["abs_error"] = (X_test_g["actual_ctr"] - X_test_g["predicted_ctr"]).abs()

worst_errors = X_test_g.sort_values("abs_error", ascending=False).head(10)
worst_errors[["position_band", "impressions_march", "main_intent",
              "actual_ctr", "predicted_ctr", "abs_error"]]


**Interpretation, filled in against the real run:**

- **Permutation importance is dominated by `avg_position_march`** (0.0799) — nearly 8x every
  other feature combined. `search_volume` contributes a little (0.0109). `category_count` barely
  registers (0.0016). `main_intent` (-0.0000) and `impressions_march` (-0.0013) are effectively
  zero or slightly negative — shuffling them doesn't hurt the model at all, meaning they carry
  no real signal in this fit. **Most of the Random Forest's edge over the baseline is coming from
  modeling position more flexibly than a 5-bucket average can, not from the other four features.**
  This directly answers the "does not reward complexity alone" check: two of the five features
  earned their place, three did not.

- **All ten worst errors are underpredictions**, not overpredictions: actual CTR is surprisingly
  high (0.05-0.08) while the model predicted near its position band's typical low value
  (~0.001-0.005). These are outlier-high performers the model can't anticipate from position,
  intent, or volume alone. They're spread across multiple position bands (1-3, 4-6, 11-20, 21+)
  and both informational and transactional intent — no single band or intent category explains
  the pattern, which suggests there's a real signal the current five features simply don't
  capture (e.g. something about the specific query or content that drives unusually high CTR
  independent of position).


## 5) Self-Check

- **Method choice, defensible without the notebook in front of me?** Random Forest over a single
  Decision Tree specifically because of documented noise (37% zero-click ties, Week 4) that a
  single tree would overfit to; over Logistic Regression and K-Means because this is a continuous,
  supervised target, not a classification or unsupervised problem.

- **Split design, defensible and demonstrated, not just asserted?** Yes — trained the identical
  model under both splits. Naive random split MAE: 0.00245. Grouped-by-client split MAE: 0.00268.
  The random split looked 9% better, which would have been a false result caused by the model
  partially learning per-client CTR levels rather than the general position-to-CTR relationship —
  exactly the failure mode the session demonstrated. The grouped split is the number used for
  every comparison. Overlap check between grouped train/test client sets: 0, confirming the split
  actually separated clients as intended.

- **Metric, defensible?** MAE, chosen for direct comparability with the frozen Week 4 baseline on
  identical rows, with the explicit tradeoff named: Precision@K would likely better fit the actual
  triage-queue use case, but was set aside here to keep this comparison apples-to-apples.

- **Does the result actually beat the baseline?** Yes, modestly: Week 4 baseline MAE 0.00277 vs.
  Random Forest MAE 0.00268 (grouped split) — a 3.2% improvement. Reported at its real size, not
  inflated.

- **Does not reward complexity alone?** Permutation importance shows only `avg_position_march`
  and, to a much smaller extent, `search_volume` carry real signal. `main_intent`,
  `impressions_march`, and `category_count` contribute close to nothing. A simpler model using
  just position (and possibly search volume) would likely capture most of this result's benefit —
  worth naming plainly rather than implying all five features are pulling their weight.

- **What do I still not know?** This model hasn't been checked for sensitivity to the click-level
  features the way the session's own paper is said to (stripping `avg_position_march` specifically
  to see how much the 3.2% improvement depends on it alone, given it dominates permutation
  importance so heavily). It also hasn't been validated across multiple grouped folds (only one
  random 75/25 client split was run) — the session noted "better in three folds, worse in two" as
  a real possibility, and a single split can't rule that out. The worst-error pattern (systematic
  underprediction of outlier-high-CTR items) also suggests a feature this notebook doesn't have
  yet — possibly something query-level — is missing from the current five. Both are reasonable
  next steps before treating this 3.2% improvement as a stable, generalizable result rather than
  a single snapshot.
